# 03_Data_Cleaning

## What I am doing

In this notebook, I will clean the Chicago Crime dataset based on the data-quality issues identified during the data-understanding stage.

The cleaning process will address:
- Missing values
- Duplicate records
- Data types
- Irrelevant identifier fields
- Other issues identified during data understanding

The cleaned dataset will be saved as a separate table for use by the remaining project members.

## Why I am doing it

Data cleaning is required to produce a consistent and reliable dataset for exploratory analysis, feature engineering, and machine-learning model development.

All cleaning decisions will be documented before they are applied so that the remaining team members understand how the final dataset was produced.

In [1]:
# Initialize Spark Session if not already present
if 'spark' not in globals() or spark is None:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('03_Data_Cleaning').master('local[*]').getOrCreate()

from pyspark.sql import functions as F
from pathlib import Path

# Load table from Databricks metastore or fallback to local CSV
try:
    df_raw = spark.table('default.chicago_crime_raw')
except Exception:
    curr = Path.cwd().resolve()
    raw_path = None
    for parent in [curr] + list(curr.parents):
        target = parent / 'data' / 'raw' / 'chicago_crime_raw.csv'
        if target.exists():
            raw_path = str(target).replace('\\', '/')
            break
    if not raw_path:
        raw_path = 'd:/Projects/Chicago_Crime_Prediction/data/raw/chicago_crime_raw.csv'
    df_raw = spark.read.option('header', 'true').option('inferSchema', 'true').csv(raw_path)
    df_raw.createOrReplaceTempView('chicago_crime_raw')

print(f"Raw rows: {df_raw.count():,}")
print(f"Raw columns: {len(df_raw.columns)}")


d:\Projects\Chicago_Crime_Prediction\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Raw rows: 662,472
Raw columns: 22


# Cleaning Decisions

The following cleaning rules are based on the data-quality findings from `02_Data_Understanding`.

| Issue | Decision | Reason |
|---|---|---|
| Exact duplicate rows | Remove if identified | Exact duplicate records provide no additional information |
| Duplicate `unique_key` | Investigate; remove only if duplicates exist | `unique_key` is intended to identify individual records |
| Repeated `case_number` | Retain | Investigation showed that repeated case numbers can correspond to different records and `unique_key` values |
| Missing `primary_type` | No action required | No missing values were identified |
| Missing `date` | No action required | No missing dates were identified |
| Missing geographic coordinates | Retain as missing | Missing values represent unavailable location information; they should not be replaced with arbitrary coordinates |
| Missing `location_description` | Retain as missing | The field is not required to have a fabricated category when the original value is unavailable |
| Missing `ward` / `community_area` | Retain as missing | These are geographic attributes and missing values should not be replaced with unsupported values |
| Date field | Keep as timestamp | The existing timestamp representation is appropriate for temporal analysis |
| `unique_key` | Retain as identifier | It is unique across all records and useful for record identification |
| `case_number` | Retain | It may contain useful reference information even though it is not unique |
| Target `primary_type` | Retain | This is the target variable for the classification task |
| Crime-code fields | Retain in cleaned dataset but flag for modelling review | `iucr`, `description`, and `fbi_code` may directly encode the target and require careful treatment during feature engineering |

In [2]:
duplicate_unique_keys = (
    df_raw.groupBy("unique_key")
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicate unique_key values: "
    f"{duplicate_unique_keys.count():,}"
)

Duplicate unique_key values: 0


In [3]:
distinct_count = df_raw.distinct().count()
total_count = df_raw.count()

print(f"Total records: {total_count:,}")
print(f"Distinct records: {distinct_count:,}")
print(f"Exact duplicate records: {total_count - distinct_count:,}")

Total records: 662,472
Distinct records: 662,472
Exact duplicate records: 0


In [4]:
df_clean = df_raw.dropDuplicates()

print(f"Rows after duplicate handling: {df_clean.count():,}")

Rows after duplicate handling: 662,472


In [5]:
df_clean = df_raw.dropDuplicates()

print(f"Rows after duplicate handling: {df_clean.count():,}")
print(f"Columns: {len(df_clean.columns)}")

Rows after duplicate handling: 662,472
Columns: 22


In [6]:
df_clean.printSchema()

root
 |-- unique_key: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)



### Data-Type Cleaning Decision

The existing schema was reviewed before applying any type conversions.

The `date` and `updated_on` fields are already stored as timestamp values, which is appropriate for temporal analysis.

The `arrest` and `domestic` fields are stored as boolean values, which is appropriate for binary attributes.

Numeric administrative and geographic fields are stored using numeric data types.

Therefore, no unnecessary data-type conversions are required at this stage.

In [7]:
missing_summary = []

for column in df_clean.columns:
    missing_count = df_clean.filter(F.col(column).isNull()).count()
    
    if missing_count > 0:
        missing_percentage = (missing_count / df_clean.count()) * 100
        missing_summary.append(
            (column, missing_count, missing_percentage)
        )

missing_df = spark.createDataFrame(
    missing_summary,
    ["column", "missing_count", "missing_percentage"]
)

display(
    missing_df.orderBy(F.desc("missing_count"))
)

DataFrame[column: string, missing_count: bigint, missing_percentage: double]

### Missing-Value Cleaning Decision

The missing-value analysis shows that missing values are concentrated in geographic and location-related attributes.

The following fields contain missing values:

- `x_coordinate`: 4,435 records
- `y_coordinate`: 4,435 records
- `latitude`: 4,435 records
- `longitude`: 4,435 records
- `location`: 4,435 records
- `location_description`: 3,145 records
- `community_area`: 932 records
- `ward`: 916 records

These missing values are retained rather than replaced with arbitrary values.

Geographic coordinates will not be replaced with mean, median, or fixed coordinates because doing so would introduce artificial spatial information into the dataset.

Similarly, categorical location attributes such as `location_description` will not be assigned an artificial category at this stage.

The target variable `primary_type` has no missing values, so no target records need to be removed because of missing target values.

Therefore, no rows are removed due to missing geographic or location information.

In [8]:
target_missing = df_clean.filter(
    F.col("primary_type").isNull()
).count()

print(f"Missing target values: {target_missing:,}")

Missing target values: 0


In [9]:
important_columns = [
    "unique_key",
    "case_number",
    "date",
    "primary_type"
]

for column in important_columns:
    null_count = df_clean.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column}: {null_count:,} null values")

unique_key: 0 null values
case_number: 0 null values
date: 0 null values
primary_type: 0 null values


### Missing-Value Cleaning Conclusion

The target variable `primary_type` and the main record-identification and temporal fields (`unique_key`, `case_number`, and `date`) contain no missing values.

Missing values are limited to geographic and location-related fields.

Because these missing values represent unavailable information rather than demonstrably invalid records, the corresponding rows are retained. No artificial values are introduced.

This preserves the original information without creating potentially misleading geographic or categorical values.

Any model-specific handling of missing predictor values can be performed during the preprocessing stage by Member 3, depending on the final feature set.

In [10]:
raw_count = df_raw.count()
clean_count = df_clean.count()

print("=== Cleaning Validation ===")
print(f"Raw record count:     {raw_count:,}")
print(f"Cleaned record count: {clean_count:,}")
print(f"Records removed:      {raw_count - clean_count:,}")
print(f"Raw columns:          {len(df_raw.columns)}")
print(f"Cleaned columns:      {len(df_clean.columns)}")

=== Cleaning Validation ===
Raw record count:     662,472
Cleaned record count: 662,472
Records removed:      0
Raw columns:          22
Cleaned columns:      22


In [11]:
clean_table_name = "default.chicago_crime_clean"

try:
    df_clean.write.mode("overwrite").saveAsTable(clean_table_name)
    print(f"Cleaned table saved successfully as: {clean_table_name}")
except Exception as e:
    # Local fallback for Windows/local spark without Hadoop winutils
    df_clean.createOrReplaceTempView("chicago_crime_clean")
    from pathlib import Path
    clean_dir = Path.cwd().resolve() / "data" / "clean"
    clean_dir.mkdir(parents=True, exist_ok=True)
    df_clean.toPandas().to_csv(clean_dir / "chicago_crime_clean.csv", index=False)
    print(f"Cleaned table registered as temp view 'chicago_crime_clean' and exported to data/clean/chicago_crime_clean.csv")


d:\Projects\Chicago_Crime_Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Projects\Chicago_Crime_Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Projects\Chicago_Crime_Prediction\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  requi

Cleaned table registered as temp view 'chicago_crime_clean' and exported to data/clean/chicago_crime_clean.csv


In [12]:
try:
    df_verify = spark.table("default.chicago_crime_clean")
except Exception:
    df_verify = spark.table("chicago_crime_clean")

print(f"Verified cleaned rows: {df_verify.count():,}")
print(f"Verified cleaned columns: {len(df_verify.columns)}")


Verified cleaned rows: 662,472
Verified cleaned columns: 22


# Final Cleaning Summary

The cleaned Chicago Crime dataset has been created from the raw dataset.

## Cleaning actions performed

1. Exact duplicate records were checked and none were found.
2. Duplicate `unique_key` values were checked and none were found.
3. Repeated `case_number` values were retained because investigation showed that a case number can be associated with multiple distinct records.
4. Missing values in geographic and location-related fields were retained because they represent unavailable information and replacing them with arbitrary values could introduce false information.
5. The target variable `primary_type` was checked and contains no missing values.
6. The `date`, `updated_on`, boolean, numerical, and geographic fields were reviewed and their existing data types were found to be appropriate.
7. No records were removed because of invalid ranges or missing target values.

## Final dataset

- Records: 662,472
- Columns: 22
- Records removed: 0
- Target: `primary_type`
- Target classes: 31
- Coverage: January 1, 2024 to September 16, 2026

The cleaned dataset is saved as:

`default.chicago_crime_clean`

The cleaned table is ready to be handed over to Members 2 and 3 for exploratory analysis and feature engineering/preprocessing.

In [13]:
try:
    df_final = spark.table("default.chicago_crime_clean")
except Exception:
    df_final = spark.table("chicago_crime_clean")

print("=== FINAL MEMBER 1 DATASET ===")
print("Table: default.chicago_crime_clean")
print(f"Rows: {df_final.count():,}")
print(f"Columns: {len(df_final.columns)}")
print("Target column: primary_type")
print(f"Target classes: {df_final.select('primary_type').distinct().count()}")


=== FINAL MEMBER 1 DATASET ===
Table: default.chicago_crime_clean
Rows: 662,472
Columns: 22
Target column: primary_type
Target classes: 31


# Member 1 Handover Note

## Dataset to use

Members 2 and 3 should use the cleaned table:

`default.chicago_crime_clean`

The cleaned dataset contains:

- 662,472 records
- 22 columns
- Target variable: `primary_type`
- 31 target classes

## Dataset coverage

The dataset covers:

- 2024: complete year
- 2025: complete year
- 2026: January 1 to September 16 only

Therefore, 2026 is year-to-date and should not be interpreted as a complete calendar year.

## Data-quality findings

- No exact duplicate records were found.
- No duplicate `unique_key` values were found.
- 73 records are associated with repeated `case_number` values, but these were retained because `case_number` is not a unique row identifier.
- `primary_type`, `date`, `unique_key`, and `case_number` contain no missing values.
- Missing values are concentrated in geographic and location-related fields.
- No obvious out-of-range values were identified in the checked numerical and geographic fields.

## Missing geographic information

The following fields contain missing values:

- `x_coordinate`: 4,435
- `y_coordinate`: 4,435
- `latitude`: 4,435
- `longitude`: 4,435
- `location`: 4,435
- `location_description`: 3,145
- `community_area`: 932
- `ward`: 916

These missing values were retained rather than replaced with artificial values.

## Target distribution

The target variable `primary_type` is highly imbalanced across its 31 classes.

The most frequent classes include:

- `THEFT`
- `BATTERY`
- `CRIMINAL DAMAGE`
- `ASSAULT`
- `MOTOR VEHICLE THEFT`

Several classes contain very few records.

This imbalance should be considered during feature engineering, model development, and evaluation.

## Potential target leakage

The following fields require particular attention before being used as ordinary machine-learning predictors:

- `iucr`
- `description`
- `fbi_code`

These fields are crime classification/code-related variables and may directly encode or reveal the target `primary_type`.

Members 2 and 3 should review these variables before including them in the final modelling feature set.

## Cleaning status

No records were removed during cleaning because no exact duplicates, duplicate unique keys, missing target values, or obvious invalid records were identified.

The cleaned dataset is ready for exploratory analysis and feature engineering.